In [1]:
# =========================================================
# POST-PATH-CHANGE INTEGRITY CHECK — SETUP
# =========================================================

from pathlib import Path
import os
import re
import duckdb
import pandas as pd
import nbformat


def resolve_project_root():

    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root

    cwd = Path.cwd().resolve()

    for candidate in [cwd, cwd.parent]:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw" / "r6.2"
PROCESSED = DATA / "processed"
FEATURES = DATA / "features"
RESULTS = PROJECT_ROOT / "results"
NOTEBOOKS = PROJECT_ROOT / "notebooks"


CHECKS = []


def check(name, condition, details=""):

    passed = bool(condition)

    CHECKS.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "details": str(details)
    })

    icon = "✅" if passed else "❌"

    print(
        f"{icon} {name}"
        + (
            f" — {details}"
            if details
            else ""
        )
    )


print("=" * 100)
print("POST-PATH-CHANGE INTEGRITY CHECK")
print("=" * 100)

print("Project root:")
print(PROJECT_ROOT)

print(
    "\n✅ SUCCESS — Integrity-check environment ready."
)

POST-PATH-CHANGE INTEGRITY CHECK
Project root:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Integrity-check environment ready.


In [3]:
# =========================================================
# RAW DATASET SANITY
# =========================================================

raw_expectations = {
    "http.csv": 80000,
    "email.csv": 7000,
    "file.csv": 1200,
    "logon.csv": 200,
    "device.csv": 100
}


raw_rows = []

for filename, minimum_mb in raw_expectations.items():

    path = RAW / filename

    exists = path.exists()

    size_mb = (
        path.stat().st_size / (1024 ** 2)
        if exists
        else 0
    )

    raw_rows.append({
        "file": filename,
        "exists": exists,
        "size_mb": round(size_mb, 2),
        "minimum_expected_mb": minimum_mb
    })

    check(
        f"Raw file: {filename}",
        exists and size_mb >= minimum_mb,
        f"{size_mb:.2f} MB"
    )


answer_files = [
    "insiders.csv",
    "r6.2-1.csv",
    "r6.2-2.csv",
    "r6.2-3.csv",
    "r6.2-4.csv",
    "r6.2-5.csv"
]

for filename in answer_files:

    path = RAW / "answers" / filename

    check(
        f"Answer file: {filename}",
        path.exists()
    )


raw_summary = pd.DataFrame(raw_rows)

print("\n" + "=" * 100)
print("RAW FILE SUMMARY")
print("=" * 100)

print(
    raw_summary.to_string(index=False)
)

print(
    "\n✅ SUCCESS — Raw dataset sanity check completed."
)

✅ Raw file: http.csv — 86053.24 MB
✅ Raw file: email.csv — 7710.23 MB
✅ Raw file: file.csv — 1269.62 MB
✅ Raw file: logon.csv — 230.45 MB
✅ Raw file: device.csv — 133.08 MB
✅ Answer file: insiders.csv
✅ Answer file: r6.2-1.csv
✅ Answer file: r6.2-2.csv
✅ Answer file: r6.2-3.csv
✅ Answer file: r6.2-4.csv
✅ Answer file: r6.2-5.csv

RAW FILE SUMMARY
      file  exists  size_mb  minimum_expected_mb
  http.csv    True 86053.24                80000
 email.csv    True  7710.23                 7000
  file.csv    True  1269.62                 1200
 logon.csv    True   230.45                  200
device.csv    True   133.08                  100

✅ SUCCESS — Raw dataset sanity check completed.


In [5]:
# =========================================================
# FINAL DATASET ROW / COLUMN CONTRACTS
# =========================================================

dataset_contracts = {

    "email_daily_r62": {
        "path": PROCESSED / "email_daily_r62.parquet",
        "rows": 1_381_557,
        "columns": 13
    },

    "usb_daily_r62": {
        "path": PROCESSED / "usb_daily_r62.parquet",
        "rows": 308_647,
        "columns": 11
    },

    "http_daily_r62": {
        "path": PROCESSED / "http_daily_r62.parquet",
        "rows": 1_391_239,
        "columns": 8
    },

    "logon_daily_r62": {
        "path": PROCESSED / "logon_daily_r62.parquet",
        "rows": 1_394_010,
        "columns": 6
    },

    "user_day_r62_v1": {
        "path": PROCESSED / "user_day_r62_v1.parquet",
        "rows": 1_394_010,
        "columns": 34
    },

    "user_baseline_r62_v1": {
        "path": FEATURES / "user_baseline_r62_v1.parquet",
        "rows": 3_995,
        "columns": 36
    },

    "behavior_features_r62_v1": {
        "path": FEATURES / "behavior_features_r62_v1.parquet",
        "rows": 387_162,
        "columns": 85
    },

    "attack_host_eligibility_r62": {
        "path": FEATURES / "attack_host_eligibility_r62.parquet",
        "rows": 3_995,
        "columns": 9
    },

    "ground_truth_users_r62": {
        "path": PROCESSED / "ground_truth_users_r62.parquet",
        "rows": 5,
        "columns": 12
    },

    "ground_truth_events_r62": {
        "path": PROCESSED / "ground_truth_events_r62.parquet",
        "rows": 470,
        "columns": 23
    },

    "malicious_pre_attack_baseline_r62": {
        "path": FEATURES / "malicious_pre_attack_baseline_r62.parquet",
        "rows": 5,
        "columns": 36
    }
}


contract_rows = []

con = duckdb.connect()


for dataset, contract in dataset_contracts.items():

    path = contract["path"]

    if not path.exists():

        contract_rows.append({
            "dataset": dataset,
            "rows": None,
            "expected_rows": contract["rows"],
            "columns": None,
            "expected_columns": contract["columns"],
            "status": "FAIL"
        })

        check(
            f"{dataset} exists",
            False
        )

        continue


    sql_path = path.as_posix()


    row_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{sql_path}')
    """).fetchone()[0]


    column_count = len(
        con.execute(f"""
            DESCRIBE
            SELECT *
            FROM read_parquet('{sql_path}')
        """).df()
    )


    passed = (
        row_count == contract["rows"]
        and
        column_count == contract["columns"]
    )


    contract_rows.append({
        "dataset": dataset,
        "rows": row_count,
        "expected_rows": contract["rows"],
        "columns": column_count,
        "expected_columns": contract["columns"],
        "status": "PASS" if passed else "FAIL"
    })


    check(
        dataset,
        passed,
        (
            f"rows={row_count:,}, "
            f"columns={column_count}"
        )
    )


con.close()


contract_df = pd.DataFrame(
    contract_rows
)


print("\n" + "=" * 110)
print("DATASET CONTRACT SUMMARY")
print("=" * 110)

print(
    contract_df.to_string(
        index=False
    )
)

✅ email_daily_r62 — rows=1,381,557, columns=13
✅ usb_daily_r62 — rows=308,647, columns=11
✅ http_daily_r62 — rows=1,391,239, columns=8
✅ logon_daily_r62 — rows=1,394,010, columns=6
✅ user_day_r62_v1 — rows=1,394,010, columns=34
✅ user_baseline_r62_v1 — rows=3,995, columns=36
✅ behavior_features_r62_v1 — rows=387,162, columns=85
✅ attack_host_eligibility_r62 — rows=3,995, columns=9
✅ ground_truth_users_r62 — rows=5, columns=12
✅ ground_truth_events_r62 — rows=470, columns=23
✅ malicious_pre_attack_baseline_r62 — rows=5, columns=36

DATASET CONTRACT SUMMARY
                          dataset    rows  expected_rows  columns  expected_columns status
                  email_daily_r62 1381557        1381557       13                13   PASS
                    usb_daily_r62  308647         308647       11                11   PASS
                   http_daily_r62 1391239        1391239        8                 8   PASS
                  logon_daily_r62 1394010        1394010        6         

In [7]:
# =========================================================
# DAILY CHANNEL CONTENT VALIDATION
# =========================================================

con = duckdb.connect()


# ---------------------------------------------------------
# EMAIL
# ---------------------------------------------------------

email_path = (
    PROCESSED /
    "email_daily_r62.parquet"
).as_posix()

email = con.execute(f"""
SELECT

    COUNT(DISTINCT user) AS users,

    MIN(day) AS first_day,
    MAX(day) AS last_day,

    SUM(email_event_count) AS events,
    SUM(email_send_count) AS sends,
    SUM(email_view_count) AS views,

    SUM(external_email_count) AS external_emails,
    SUM(internal_email_count) AS internal_emails,

    SUM(attachment_count) AS attachments

FROM read_parquet('{email_path}')
""").fetchone()


check(
    "Email users",
    email[0] == 4000,
    email[0]
)

check(
    "Email date range",
    str(email[1]) == "2010-01-02"
    and
    str(email[2]) == "2011-05-31",
    f"{email[1]} -> {email[2]}"
)

check(
    "Email total events",
    int(email[3]) == 10_994_957,
    int(email[3])
)

check(
    "Email send/view totals",
    int(email[4]) == 3_832_466
    and
    int(email[5]) == 7_162_491,
    f"send={int(email[4])}, view={int(email[5])}"
)

check(
    "External/Internal email totals",
    int(email[6]) == 1_656_078
    and
    int(email[7]) == 2_176_388,
    f"external={int(email[6])}, internal={int(email[7])}"
)

check(
    "Email attachment total",
    int(email[8]) == 1_719_195,
    int(email[8])
)


# ---------------------------------------------------------
# USB / FILE
# ---------------------------------------------------------

usb_path = (
    PROCESSED /
    "usb_daily_r62.parquet"
).as_posix()

usb = con.execute(f"""
SELECT

    COUNT(DISTINCT user),

    SUM(file_event_count),
    SUM(file_copy_count),
    SUM(file_open_count),
    SUM(file_write_count),
    SUM(file_delete_count),

    SUM(usb_copy_count),
    SUM(from_removable_count)

FROM read_parquet('{usb_path}')
""").fetchone()


check(
    "USB/File users",
    usb[0] == 3185,
    usb[0]
)

check(
    "File event total",
    int(usb[1]) == 2_014_883,
    int(usb[1])
)

check(
    "File activity totals",
    (
        int(usb[2]) == 725_458
        and int(usb[3]) == 657_500
        and int(usb[4]) == 251_126
        and int(usb[5]) == 380_799
    ),
    (
        f"copy={int(usb[2])}, "
        f"open={int(usb[3])}, "
        f"write={int(usb[4])}, "
        f"delete={int(usb[5])}"
    )
)

check(
    "USB copy total",
    int(usb[6]) == 333_611,
    int(usb[6])
)

check(
    "From-removable total",
    int(usb[7]) == 971_633,
    int(usb[7])
)


# ---------------------------------------------------------
# HTTP
# ---------------------------------------------------------

http_path = (
    PROCESSED /
    "http_daily_r62.parquet"
).as_posix()

http = con.execute(f"""
SELECT

    COUNT(DISTINCT user),

    SUM(http_event_count),
    SUM(www_visit_count),
    SUM(www_download_count),
    SUM(www_upload_count)

FROM read_parquet('{http_path}')
""").fetchone()


check(
    "HTTP users",
    http[0] == 4000,
    http[0]
)

check(
    "HTTP event total",
    int(http[1]) == 117_025_216,
    int(http[1])
)

check(
    "HTTP activity totals",
    (
        int(http[2]) == 115_401_178
        and int(http[3]) == 1_413_637
        and int(http[4]) == 210_401
    ),
    (
        f"visit={int(http[2])}, "
        f"download={int(http[3])}, "
        f"upload={int(http[4])}"
    )
)


# ---------------------------------------------------------
# LOGON
# ---------------------------------------------------------

logon_path = (
    PROCESSED /
    "logon_daily_r62.parquet"
).as_posix()

logon = con.execute(f"""
SELECT

    COUNT(DISTINCT user),

    MIN(day),
    MAX(day),

    SUM(logon_count),
    SUM(logoff_count)

FROM read_parquet('{logon_path}')
""").fetchone()


check(
    "Logon users",
    logon[0] == 4000,
    logon[0]
)

check(
    "Logon date range",
    str(logon[1]) == "2010-01-02"
    and
    str(logon[2]) == "2011-06-01",
    f"{logon[1]} -> {logon[2]}"
)

check(
    "Logon/Logoff totals",
    (
        int(logon[3]) == 1_948_933
        and
        int(logon[4]) == 1_581_352
    ),
    (
        f"logon={int(logon[3])}, "
        f"logoff={int(logon[4])}"
    )
)


con.close()

print(
    "\n✅ SUCCESS — Daily channel content validation completed."
)

✅ Email users — 4000
✅ Email date range — 2010-01-02 -> 2011-05-31
✅ Email total events — 10994957
✅ Email send/view totals — send=3832466, view=7162491
✅ External/Internal email totals — external=1656078, internal=2176388
✅ Email attachment total — 1719195
✅ USB/File users — 3185
✅ File event total — 2014883
✅ File activity totals — copy=725458, open=657500, write=251126, delete=380799
✅ USB copy total — 333611
✅ From-removable total — 971633
✅ HTTP users — 4000
✅ HTTP event total — 117025216
✅ HTTP activity totals — visit=115401178, download=1413637, upload=210401
✅ Logon users — 4000
✅ Logon date range — 2010-01-02 -> 2011-06-01
✅ Logon/Logoff totals — logon=1948933, logoff=1581352

✅ SUCCESS — Daily channel content validation completed.


In [9]:
# =========================================================
# UNIFIED USER-DAY INTEGRITY
# =========================================================

user_day_path = (
    PROCESSED /
    "user_day_r62_v1.parquet"
).as_posix()

con = duckdb.connect()


summary = con.execute(f"""
SELECT

    COUNT(*) AS rows,
    COUNT(DISTINCT user) AS users,
    COUNT(DISTINCT day) AS calendar_days,

    MIN(day) AS first_day,
    MAX(day) AS last_day,

    SUM(email_event_count) AS email_events,
    SUM(usb_copy_count) AS usb_copies,
    SUM(http_event_count) AS http_events,
    SUM(www_upload_count) AS http_uploads,

    SUM(logon_count) AS logons,
    SUM(logoff_count) AS logoffs

FROM read_parquet('{user_day_path}')
""").fetchone()


duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT
        user,
        day,
        COUNT(*) AS n

    FROM read_parquet('{user_day_path}')

    GROUP BY user, day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


con.close()


check(
    "User-Day rows",
    summary[0] == 1_394_010,
    summary[0]
)

check(
    "User-Day users",
    summary[1] == 4000,
    summary[1]
)

check(
    "User-Day calendar days",
    summary[2] == 516,
    summary[2]
)

check(
    "User-Day date range",
    (
        str(summary[3]) == "2010-01-02"
        and
        str(summary[4]) == "2011-06-01"
    ),
    f"{summary[3]} -> {summary[4]}"
)

check(
    "User-Day email total preserved",
    int(summary[5]) == 10_994_957,
    int(summary[5])
)

check(
    "User-Day USB total preserved",
    int(summary[6]) == 333_611,
    int(summary[6])
)

check(
    "User-Day HTTP total preserved",
    int(summary[7]) == 117_025_216
    and
    int(summary[8]) == 210_401,
    (
        f"events={int(summary[7])}, "
        f"uploads={int(summary[8])}"
    )
)

check(
    "User-Day Logon totals preserved",
    int(summary[9]) == 1_948_933
    and
    int(summary[10]) == 1_581_352
)

check(
    "User-Day duplicate keys",
    duplicates == 0,
    duplicates
)

print(
    "\n✅ SUCCESS — Unified User-Day integrity validated."
)

✅ User-Day rows — 1394010
✅ User-Day users — 4000
✅ User-Day calendar days — 516
✅ User-Day date range — 2010-01-02 -> 2011-06-01
✅ User-Day email total preserved — 10994957
✅ User-Day USB total preserved — 333611
✅ User-Day HTTP total preserved — events=117025216, uploads=210401
✅ User-Day Logon totals preserved
✅ User-Day duplicate keys

✅ SUCCESS — Unified User-Day integrity validated.


In [11]:
# =========================================================
# BASELINE + BEHAVIOR INTEGRITY
# =========================================================

malicious_users = [
    "ACM2278",
    "CMP2946",
    "PLJ1771",
    "CDE1846",
    "MBG3183"
]

malicious_sql = ", ".join(
    f"'{u}'"
    for u in malicious_users
)


baseline_path = (
    FEATURES /
    "user_baseline_r62_v1.parquet"
).as_posix()

behavior_path = (
    FEATURES /
    "behavior_features_r62_v1.parquet"
).as_posix()


con = duckdb.connect()


# ---------------------------------------------------------
# BASELINE
# ---------------------------------------------------------

baseline = con.execute(f"""
SELECT

    COUNT(*),
    COUNT(DISTINCT user),

    MIN(baseline_workdays),
    MEDIAN(baseline_workdays),
    MAX(baseline_workdays),

    SUM(
        CASE
            WHEN email_active_days >= 50
             AND usb_active_days >= 50
             AND http_upload_active_days >= 50
            THEN 1
            ELSE 0
        END
    )

FROM read_parquet('{baseline_path}')
""").fetchone()


malicious_in_baseline = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet('{baseline_path}')

WHERE user IN ({malicious_sql})
""").fetchone()[0]


check(
    "Benign baseline population",
    baseline[0] == 3995
    and baseline[1] == 3995,
    f"rows={baseline[0]}, users={baseline[1]}"
)

check(
    "Baseline workday distribution",
    (
        baseline[2] == 21
        and float(baseline[3]) == 251.0
        and baseline[4] == 260
    ),
    (
        f"min={baseline[2]}, "
        f"median={baseline[3]}, "
        f"max={baseline[4]}"
    )
)

check(
    "Baseline >=50 days all channels",
    int(baseline[5]) == 131,
    int(baseline[5])
)

check(
    "No malicious users in benign baseline",
    malicious_in_baseline == 0,
    malicious_in_baseline
)


# ---------------------------------------------------------
# BEHAVIOR FEATURES
# ---------------------------------------------------------

behavior = con.execute(f"""
SELECT

    COUNT(*),
    COUNT(DISTINCT user),

    MIN(day),
    MAX(day),

    MIN(baseline_workdays),
    MEDIAN(baseline_workdays),
    MAX(baseline_workdays)

FROM read_parquet('{behavior_path}')
""").fetchone()


behavior_duplicates = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT user, day, COUNT(*) AS n

    FROM read_parquet('{behavior_path}')

    GROUP BY user, day

    HAVING COUNT(*) > 1
)
""").fetchone()[0]


behavior_malicious = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet('{behavior_path}')

WHERE user IN ({malicious_sql})
""").fetchone()[0]


critical_nulls = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet('{behavior_path}')

WHERE user IS NULL
   OR day IS NULL
   OR baseline_workdays IS NULL
   OR external_email_count IS NULL
   OR usb_copy_count IS NULL
   OR www_upload_count IS NULL
   OR active_channel_count IS NULL
""").fetchone()[0]


check(
    "Behavior population",
    behavior[0] == 387_162
    and behavior[1] == 3737,
    f"rows={behavior[0]}, users={behavior[1]}"
)

check(
    "Behavior date range",
    (
        str(behavior[2]) == "2011-01-03"
        and
        str(behavior[3]) == "2011-06-01"
    ),
    f"{behavior[2]} -> {behavior[3]}"
)

check(
    "Behavior baseline-workday distribution",
    (
        behavior[4] == 249
        and float(behavior[5]) == 251.0
        and behavior[6] == 260
    ),
    (
        f"min={behavior[4]}, "
        f"median={behavior[5]}, "
        f"max={behavior[6]}"
    )
)

check(
    "Behavior duplicate user-days",
    behavior_duplicates == 0,
    behavior_duplicates
)

check(
    "No malicious users in behavior dataset",
    behavior_malicious == 0,
    behavior_malicious
)

check(
    "Behavior critical NULLs",
    critical_nulls == 0,
    critical_nulls
)


con.close()

print(
    "\n✅ SUCCESS — Baseline and behavior integrity validated."
)

✅ Benign baseline population — rows=3995, users=3995
✅ Baseline workday distribution — min=21, median=251.0, max=260
✅ Baseline >=50 days all channels — 131
✅ No malicious users in benign baseline
✅ Behavior population — rows=387162, users=3737
✅ Behavior date range — 2011-01-03 -> 2011-06-01
✅ Behavior baseline-workday distribution — min=249, median=251.0, max=260
✅ Behavior duplicate user-days
✅ No malicious users in behavior dataset
✅ Behavior critical NULLs

✅ SUCCESS — Baseline and behavior integrity validated.


In [13]:
# =========================================================
# ATTACK HOSTS + GROUND TRUTH + MALICIOUS BASELINE
# =========================================================

attack_path = (
    FEATURES /
    "attack_host_eligibility_r62.parquet"
).as_posix()

gt_users_path = (
    PROCESSED /
    "ground_truth_users_r62.parquet"
).as_posix()

gt_events_path = (
    PROCESSED /
    "ground_truth_events_r62.parquet"
).as_posix()

mal_baseline_path = (
    FEATURES /
    "malicious_pre_attack_baseline_r62.parquet"
).as_posix()


con = duckdb.connect()


# ---------------------------------------------------------
# ATTACK HOSTS
# ---------------------------------------------------------

attack = con.execute(f"""
SELECT

    COUNT(*),
    SUM(eligible_30),
    SUM(eligible_40),
    SUM(eligible_strict_50)

FROM read_parquet('{attack_path}')
""").fetchone()


check(
    "Attack-host population",
    attack[0] == 3995,
    attack[0]
)

check(
    "Attack-host eligibility counts",
    (
        int(attack[1]) == 199
        and
        int(attack[2]) == 158
        and
        int(attack[3]) == 126
    ),
    (
        f"30={int(attack[1])}, "
        f"40={int(attack[2])}, "
        f"50={int(attack[3])}"
    )
)


# ---------------------------------------------------------
# GROUND TRUTH USERS
# ---------------------------------------------------------

gt_users = con.execute(f"""
SELECT

    COUNT(*),
    COUNT(DISTINCT user),
    COUNT(DISTINCT scenario),
    SUM(start_was_corrected)

FROM read_parquet('{gt_users_path}')
""").fetchone()


check(
    "Ground-truth malicious users",
    (
        gt_users[0] == 5
        and
        gt_users[1] == 5
        and
        gt_users[2] == 5
        and
        int(gt_users[3]) == 1
    ),
    (
        f"rows={gt_users[0]}, "
        f"users={gt_users[1]}, "
        f"scenarios={gt_users[2]}"
    )
)


# ---------------------------------------------------------
# GROUND TRUTH EVENTS
# ---------------------------------------------------------

gt_events = con.execute(f"""
SELECT

    COUNT(*),
    COUNT(DISTINCT event_id),
    SUM(is_primary_actor_event),
    SUM(is_context_event),
    COUNT(DISTINCT malicious_actor)

FROM read_parquet('{gt_events_path}')
""").fetchone()


check(
    "Ground-truth event population",
    (
        gt_events[0] == 470
        and
        gt_events[1] == 470
    ),
    (
        f"events={gt_events[0]}, "
        f"unique_ids={gt_events[1]}"
    )
)

check(
    "Ground-truth event roles",
    (
        int(gt_events[2]) == 420
        and
        int(gt_events[3]) == 50
        and
        gt_events[4] == 5
    ),
    (
        f"primary={int(gt_events[2])}, "
        f"context={int(gt_events[3])}"
    )
)


# ---------------------------------------------------------
# MALICIOUS PRE-ATTACK BASELINES
# ---------------------------------------------------------

mal_baseline = con.execute(f"""
SELECT

    COUNT(*),
    COUNT(DISTINCT user),

    MIN(baseline_workdays),
    MEDIAN(baseline_workdays),
    MAX(baseline_workdays),

    SUM(
        CASE
            WHEN baseline_last_day
                 >= CAST(attack_start AS DATE)
            THEN 1
            ELSE 0
        END
    )

FROM read_parquet('{mal_baseline_path}')
""").fetchone()


check(
    "Malicious pre-attack baseline population",
    mal_baseline[0] == 5
    and
    mal_baseline[1] == 5,
    f"rows={mal_baseline[0]}, users={mal_baseline[1]}"
)

check(
    "Malicious baseline workday distribution",
    (
        mal_baseline[2] == 155
        and float(mal_baseline[3]) == 197.0
        and mal_baseline[4] == 286
    ),
    (
        f"min={mal_baseline[2]}, "
        f"median={mal_baseline[3]}, "
        f"max={mal_baseline[4]}"
    )
)

check(
    "Malicious baseline temporal leakage",
    int(mal_baseline[5]) == 0,
    int(mal_baseline[5])
)


con.close()

print(
    "\n✅ SUCCESS — Attack-host and ground-truth integrity validated."
)

✅ Attack-host population — 3995
✅ Attack-host eligibility counts — 30=199, 40=158, 50=126
✅ Ground-truth malicious users — rows=5, users=5, scenarios=5
✅ Ground-truth event population — events=470, unique_ids=470
✅ Ground-truth event roles — primary=420, context=50
✅ Malicious pre-attack baseline population — rows=5, users=5
✅ Malicious baseline workday distribution — min=155, median=197.0, max=286
✅ Malicious baseline temporal leakage

✅ SUCCESS — Attack-host and ground-truth integrity validated.


In [15]:
# =========================================================
# DOCUMENTATION INTEGRITY
# =========================================================

feature_dictionary_path = (
    RESULTS /
    "feature_dictionary_r62.csv"
)


check(
    "Feature Dictionary exists",
    feature_dictionary_path.exists()
)


if feature_dictionary_path.exists():

    feature_dictionary = pd.read_csv(
        feature_dictionary_path
    )

    check(
        "Feature Dictionary rows",
        len(feature_dictionary) == 85,
        len(feature_dictionary)
    )

    check(
        "Feature Dictionary unique features",
        feature_dictionary["feature_name"].nunique() == 85,
        feature_dictionary["feature_name"].nunique()
    )


    required_fields = [
        "feature_name",
        "category",
        "feature_role",
        "description",
        "calculation",
        "time_scope",
        "source_columns"
    ]


    complete_metadata = all(
        field in feature_dictionary.columns
        and
        feature_dictionary[field]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .all()

        for field in required_fields
    )


    check(
        "Feature Dictionary semantic metadata",
        complete_metadata
    )


# ---------------------------------------------------------
# SCHEMA DICTIONARY
# Accept current or renamed version
# ---------------------------------------------------------

new_schema = (
    RESULTS /
    "schema_dictionary_r62.csv"
)

old_schema = (
    RESULTS /
    "data_dictionary_r62.csv"
)


schema_file = (
    new_schema
    if new_schema.exists()
    else old_schema
)


check(
    "Schema Dictionary exists",
    schema_file.exists(),
    (
        schema_file.name
        if schema_file.exists()
        else "missing"
    )
)


if old_schema.exists() and not new_schema.exists():

    print(
        "\n⚠️ Naming only: data_dictionary_r62.csv "
        "still uses the old name."
    )

    print(
        "   We can rename it to "
        "schema_dictionary_r62.csv after integrity validation."
    )


print(
    "\n✅ SUCCESS — Documentation integrity check completed."
)

✅ Feature Dictionary exists
✅ Feature Dictionary rows — 85
✅ Feature Dictionary unique features — 85
✅ Feature Dictionary semantic metadata
✅ Schema Dictionary exists — schema_dictionary_r62.csv

✅ SUCCESS — Documentation integrity check completed.


In [17]:
# =========================================================
# HARD-CODED OLD PROJECT PATH AUDIT
# =========================================================

OLD_PROJECT_ROOT = (
    r"C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62"
)

# Normalize for reliable comparison
old_root_normalized = (
    OLD_PROJECT_ROOT
    .replace("\\", "/")
    .lower()
)


path_issues = []


# Notebooks intentionally excluded from this audit
# because they are not part of the final pipeline.
EXCLUDED_NOTEBOOKS = {
    "00_r62_feasibility_audit_report.ipynb",
    "09_final_validation.ipynb"
}


for notebook_path in sorted(
    NOTEBOOKS.glob("*.ipynb")
):

    if notebook_path.name in EXCLUDED_NOTEBOOKS:
        continue


    with notebook_path.open(
        "r",
        encoding="utf-8"
    ) as f:

        nb = nbformat.read(
            f,
            as_version=4
        )


    for cell_number, cell in enumerate(
        nb.cells,
        start=1
    ):

        if cell.cell_type != "code":
            continue


        for line_number, line in enumerate(
            cell.source.splitlines(),
            start=1
        ):

            normalized_line = (
                line
                .replace("\\", "/")
                .lower()
            )


            if old_root_normalized in normalized_line:

                path_issues.append({
                    "notebook":
                        notebook_path.name,

                    "cell":
                        cell_number,

                    "line":
                        line_number,

                    "code":
                        line.strip()
                })


path_issue_df = pd.DataFrame(
    path_issues
)


print("=" * 110)
print("HARD-CODED OLD PROJECT PATH AUDIT")
print("=" * 110)


if len(path_issue_df) == 0:

    print(
        "No references to the old project root "
        "were found in notebook code cells."
    )

else:

    print(
        path_issue_df.to_string(
            index=False
        )
    )


check(
    "No old hard-coded project root in notebook code",
    len(path_issues) == 0,
    f"issues={len(path_issues)}"
)


print(
    "\n✅ SUCCESS — Old project-path audit completed."
)

HARD-CODED OLD PROJECT PATH AUDIT
No references to the old project root were found in notebook code cells.
✅ No old hard-coded project root in notebook code — issues=0

✅ SUCCESS — Old project-path audit completed.


In [19]:
# =========================================================
# FINAL INTEGRITY VERDICT
# =========================================================

checks_df = pd.DataFrame(
    CHECKS
)


total_checks = len(
    checks_df
)

passed_checks = (
    checks_df["status"]
    .eq("PASS")
    .sum()
)

failed_checks = (
    checks_df["status"]
    .eq("FAIL")
    .sum()
)


print("=" * 110)
print("POST-PATH-CHANGE FINAL INTEGRITY REPORT")
print("=" * 110)

print(
    f"Total checks : {total_checks}"
)

print(
    f"Passed       : {passed_checks}"
)

print(
    f"Failed       : {failed_checks}"
)


if failed_checks > 0:

    print("\n" + "=" * 110)
    print("FAILED CHECKS")
    print("=" * 110)

    print(
        checks_df[
            checks_df[
                "status"
            ] == "FAIL"
        ].to_string(
            index=False
        )
    )

    raise AssertionError(
        "Integrity validation failed. "
        "Do not continue to the next stage yet."
    )


print("\n" + "=" * 110)

print(
    "✅ ALL INTEGRITY CHECKS PASSED"
)

print(
    "✅ PATH MIGRATION DID NOT ALTER "
    "THE VERIFIED DATA ENGINEERING OUTPUTS"
)

print(
    "✅ DATA ENGINEERING R6.2 REMAINS CONSISTENT"
)

print("=" * 110)

POST-PATH-CHANGE FINAL INTEGRITY REPORT
Total checks : 72
Passed       : 72
Failed       : 0

✅ ALL INTEGRITY CHECKS PASSED
✅ PATH MIGRATION DID NOT ALTER THE VERIFIED DATA ENGINEERING OUTPUTS
✅ DATA ENGINEERING R6.2 REMAINS CONSISTENT
